# CIS 5450 — arXiv Dataset: Team Submission

This notebook is the combined submission for the arXiv metadata analysis project.

| Section | Author | Topic |
|---------|--------|-------|
| Part 1  | King (AKing2713) | Word-frequency EDA, TF-IDF, K-Means clustering |
| Part 2  | Geon (Jion7) | LLM keyword trend tracking, ARIMA/Prophet forecasting |
| Part 3  | Ryan | Category co-occurrence network, LDA topic modeling, submission calendar patterns, author productivity |

---

---
# Part 1 — King's Analysis
*Word-frequency EDA · TF-IDF · K-Means Clustering*


# Arxiv Dataset

## Introduction

This project explores the arXiv dataset with the goal of understanding patterns in scientific research through the analysis of paper abstracts. The dataset contains metadata for a large collection of research papers, including titles, authors, categories, and abstracts. For this analysis, we focus primarily on the abstract text, as it provides a concise summary of each paper’s content and is well-suited for natural language processing (NLP) techniques.

To enable meaningful analysis, the abstract text was preprocessed using standard NLP techniques, including tokenization, stopword removal, and lemmatization. This preprocessing step transforms raw text into a structured format that can be used for both exploratory analysis and modeling.

### Project Objectives

The primary goal of this project is to investigate whether meaningful structure and patterns can be extracted from research abstracts. Specifically, this portion of the analysis focuses on the following questions:

**What are the most common terms across all abstracts?**
* This helps identify general language patterns and commonly used terminology in scientific writing.

**How does vocabulary differ across research domains?**
* By comparing term usage across arXiv categories, we can determine whether different fields exhibit distinct linguistic signatures.

**Do similar papers cluster together naturally based on their abstract content?**
* Using unsupervised learning techniques such as KMeans clustering, we evaluate whether papers group into meaningful clusters without using predefined labels.

### Methodology Overview

The analysis follows a structured pipeline:

**Data Preparation**
* Load and sample the arXiv dataset
* Extract relevant fields (e.g., id, abstract, categories)
* Preserve identifiers to maintain linkage with metadata

**Text Preprocessing**
* Lowercasing and cleaning text
* Tokenization
* Stopword removal (including domain-specific stopwords)
* Lemmatization using NLTK

**Exploratory Data Analysis (EDA)**
* Word frequency analysis
* Domain-specific vocabulary comparison using TF-IDF
* Visualization of key term distributions

**Unsupervised Learning**
* Convert text into numerical features using TF-IDF
* Apply clustering algorithms (KMeans)
* Evaluate clustering performance using multiple metrics
* Visualize clusters using dimensionality reduction techniques

Below is a Data Dictionary to aid us in the understanding of the data.

## Data Dictionary

| Column Name      | Data Type     | Description                                                                                                                                            |
| ---------------- | ------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `id`             | string        | Unique identifier assigned to each paper on arXiv (e.g., `0704.0001`). Can be used to construct URLs to the paper.                                     |
| `submitter`      | string        | Name of the individual who submitted the paper to arXiv (not necessarily the first author).                                                            |
| `authors`        | string        | Raw, comma-separated list of authors as a single text field.                                                                                           |
| `authors_parsed` | list of lists | Structured representation of authors in the format `[Last Name, First Name(s), Suffix]`.                                                               |
| `title`          | string        | Title of the research paper.                                                                                                                           |
| `abstract`       | string        | Full abstract of the paper.                                                                                                                            |
| `categories`     | string        | Space-separated list of subject categories assigned by arXiv (e.g., `cs.AI`, `math.CO`). The first category is considered the primary classification.  |
| `comments`       | string / null | Optional author-provided comments (e.g., number of pages, figures, publication notes).                                                                 |
| `journal-ref`    | string / null | Reference to the journal where the paper was published, if applicable.                                                                                 |
| `doi`            | string / null | Digital Object Identifier linking to the published version of the paper.                                                                               |
| `report-no`      | string / null | Institutional or technical report number associated with the paper.                                                                                    |
| `license`        | string / null | URL specifying the licensing terms under which the paper is distributed.                                                                               |
| `versions`       | list of dicts | List of version records, each containing version number and submission date (e.g., `v1`, `v2`).                                                        |
| `update_date`    | string (date) | Date of the most recent update to the paper in `YYYY-MM-DD` format.                                                                                    |


## Required Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import prepare
from prepare import random_sample, prepare_abstracts_for_nlp
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

## Loading Data

In [ ]:
df = pd.read_json('./arxiv-metadata-oai-snapshot.json', lines=True)

A peek at the data

In [ ]:
df.head()

Looking at the quality of the data (nulls, dtypes, etc.)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

While the data does look fairly sparse in some cases, the most important field "abstract" looks to be in good shape. Although, the other field of interest, "categories", seems to have multiple instances it can fall into. We will have to clean this up a little later in order to answer some of our questions.

## Preparing Abstracts
To prepare the abstract text for analysis, standard natural language processing (NLP) techniques were applied to clean and normalize the data. This step ensures that the text is in a consistent format and reduces noise that could negatively impact downstream analysis.

The preprocessing pipeline includes:

* Converting text to lowercase
* Removing punctuation, numerical values, and formatting artifacts (e.g., LaTeX syntax)
* Tokenizing text into individual words
* Removing common stopwords, along with domain-specific filler terms
* Lemmatizing tokens to their base form using NLTK

Both the cleaned token lists and reconstructed text were retained, allowing for flexibility in analysis (e.g., frequency counts, TF-IDF, and modeling), while preserving the original abstract text for reference.

Due to the length of the text preproccessing, the functions were moved to a .py file called "prepare.py". Please see the file for a breakdown of the functions.

In [ ]:
file_path = "abstracts_parsed_full.parquet"

if os.path.exists(file_path):
    print("Loading existing file...")
    df_abst = pd.read_parquet(file_path)
else:
    print("File not found. Creating dataset...")
    df_abst = prepare_abstracts_for_nlp(df)
    
    df_abst.to_parquet(file_path, index=False)

Lets take a look at what the text processing produced.

In [ ]:
df_abst.head()

## EDA

The first questions we would like to answer:
* What are the most common terms across all abstracts?

In [ ]:
# Flatten and combine all tokens
all_tokens = [token for tokens in df_abst['abstract_tokens_clean'] for token in tokens]

In [ ]:
# Count the most common words
word_counts = Counter(all_tokens)

top_20 = word_counts.most_common(20)

top_20

Well there seems to be a slight problem with this approach. While it does technically tell us the most common words throughout all the abstracts, these words are not that insightful due to their generic nature. One word that does stand out is "quantum" which is exciting to see if near the top. Let's remove some of the more generic words to see if we can find some more insights about the most common ones.

In [ ]:
# Removing some generic words
domain_stopwords = {
    "system", "data", "study", "propose", "also",
    "field", "two", "present", "state", "problem",
    "find", "function", "time", "provide", "large",
    "new", "one", "theory"
}

filtered_counts = Counter({word: count for word, count in word_counts.items() if word not in domain_stopwords})

top_20_filtered = filtered_counts.most_common(20)
top_20_filtered

Again, there seems to be a lot of generic words which is to be expected. Although the words that do stand out are things like "quantum", "network", and "energy". This suggests that there is a lot of research that has been done specifically in the quantum field. Let's go ahead and plot the results to get a better look at the distribution.

In [ ]:
words, counts = zip(*top_20_filtered)

plt.figure(figsize=(10, 5))
plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("Top 20 Most Common Terms in arXiv Abstracts")
plt.ylabel("Frequency")
plt.xlabel("Word")
plt.show()

Looking at the graph we can see a steady decrease in the words that are most common. Due to the size of the dataset it is unlikley we will see any significant drops.

The next question we would like to answer is:
* How does vocabulary differ across domains?

Let's first combine our abstracts data with the categories column from the original DataFrame. Since the first category is considered the main category, we will just use that one.

In [ ]:
df_eda = df.copy()

df_eda['primary_category'] = df_eda['categories'].str.split().str[0] # Take the first category
df_eda = df_eda.merge(df_abst[['id', 'abstract_clean']], on='id', how='inner')
df_eda = df_eda[['id', 'abstract_clean', 'primary_category']]

In [ ]:
# Grouping by category
category_docs = (df_eda.groupby('primary_category')['abstract_clean'].apply(lambda x: " ".join(x)))

In [ ]:
# Creating our Term Frequecy/Inverse Document Vectorizer
vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8
)

tfidf_matrix = vectorizer.fit_transform(category_docs)

feature_names = vectorizer.get_feature_names_out()

In order to get the top terms for each category we can create a function that creates a dictionary where the category is the key and the top n-terms are the items.

In [ ]:
def get_top_terms_per_category(tfidf_matrix, feature_names, category_docs, top_n=10):
    results = {}

    for i, category in enumerate(category_docs.index):
        row = tfidf_matrix[i].toarray().flatten()
        top_indices = row.argsort()[-top_n:][::-1]
        top_terms = [feature_names[idx] for idx in top_indices]

        results[category] = top_terms

    return results

top_terms = get_top_terms_per_category(tfidf_matrix, feature_names, category_docs)

In [ ]:
for category, terms in list(top_terms.items())[:5]:
    print(f"\n{category}:")
    print(", ".join(terms))

This looks a lot more promising, However, let's take a look at how many different categories we have.

In [ ]:
len(top_terms)

Due to the sheer amount of categories it will be hard to parse through them all. Let's create a function so we can quickly plot categories we are interested in.

In [ ]:
def plot_top_terms(category, tfidf_matrix, feature_names, category_docs, top_n=10):
    idx = list(category_docs.index).index(category)
    row = tfidf_matrix[idx].toarray().flatten()

    top_indices = row.argsort()[-top_n:][::-1]
    terms = [feature_names[i] for i in top_indices]
    scores = [row[i] for i in top_indices]

    plt.figure(figsize=(8, 4))
    plt.bar(terms, scores)
    plt.xticks(rotation=45)
    plt.title(f"Top Terms for {category}")
    plt.show()

# Let's take a look at the cs.AI
plot_top_terms("cs.AI", tfidf_matrix, feature_names, category_docs)

What about something a little bit more physics based.

In [ ]:
plot_top_terms("astro-ph", tfidf_matrix, feature_names, category_docs)

This seems to be a better approach as the categories really do differentiate which words we would see in papers.

### EDA Takeaways
**Scientific language is highly standardized across domains**

The most common terms across all abstracts are largely domain-agnostic, consisting of words such as “system,” “function,” “data,” and “algorithm”.

This suggests that scientific writing follows a consistent linguistic structure and many high-frequency terms reflect general research processes, rather than domain-specific concepts.

**Domain-specific vocabulary clearly differentiates fields**

Despite shared general language, TF-IDF analysis shows that each domain exhibits a distinct vocabulary signature.

For example:

* cs.AI emphasizes terms like “llm,” “policy,” and “multia”
* astro-ph highlights “galaxy,” “redshift,” and “xray”

This indicates that abstract text alone contains enough signal to distinguish research areas and different disciplines communicate using specialized terminology layered on top of common structure.

## Unsupervised Learning
To investigate whether research papers naturally group into meaningful categories based on their content, we will use unsupervised learning techniques applied to the processed abstract text. Specifically, abstracts are transformed into numerical representations using TF-IDF vectorization, capturing the importance of terms within each document relative to the overall corpus.

Using these representations, we will use KMeans clustering applied to group similar papers without using predefined labels. We will evaluate multiple values of k (number of clusters) using metrics such as inertia and silhouette score to identify an appropriate number of clusters. This allowed for a data-driven selection of cluster structure.

To better understand the results, clusters will be visualized using dimensionality reduction techniques (e.g., PCA) and plots.

The question we want to answer:

* Do similar papers cluster together naturally?

In [ ]:
df_model = df_eda.copy()

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=2000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8
)

X = vectorizer.fit_transform(df_model['abstract_clean'])

Due to the size of the data, we are going to use MiniBatchKMeans to speed it up. We lose out on a bit of precision but gain a lot of speed.

In [ ]:
k = 10
# Faster clustering
kmeans = MiniBatchKMeans(
    n_clusters=k,
    random_state=42,
    batch_size=1000,
    n_init=10
)

clusters = kmeans.fit_predict(X)
df_model['cluster'] = clusters

In [ ]:
score = silhouette_score(X, clusters, sample_size=5000, random_state=42)
print("Silhouette Score:", score)

The low silhouette score indicates that abstract-based clustering does not produce strongly separated groups. This suggests that research domains exhibit significant semantic overlap and that traditional bag-of-words representations (TF-IDF) may not fully capture deeper relationships between documents. While this is not the outcome we would of hoped for I wouldn't call it a failure. Let's plot of results using PCA to see if we can visually inspect the clusters.

Again, due to the size of the data we will only PCA and plot a sample of it

In [ ]:
# sample rows for plotting
n_plot = 5000
rng = np.random.default_rng(42)
idx = rng.choice(X.shape[0], size=min(n_plot, X.shape[0]), replace=False)

X_sample = X[idx].toarray()
clusters_sample = df_model['cluster'].iloc[idx]

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sample)

plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_sample, s=10, alpha=0.7)
plt.title("KMeans Clusters (PCA Projection)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

Even though our Silhouete Score is low, we do see some natural clusters forming. Let's take a look at the top terms per cluster. We will create another function similar to our one from our categories exploration.

In [ ]:
terms = vectorizer.get_feature_names_out()

def top_terms_per_cluster(kmeans, terms, n_terms=10):
    for i, center in enumerate(kmeans.cluster_centers_):
        top_indices = center.argsort()[-n_terms:][::-1]
        top_words = [terms[idx] for idx in top_indices]
        print(f"\nCluster {i}:")
        print(", ".join(top_words))

top_terms_per_cluster(kmeans, terms)

Again, though our Silhouete Score is low, there clearly seems to be different terms in different clusters. Let's go ahead and try to see if we can find a good number of clusters.

In [ ]:
k_values = [5, 10, 15, 20]
results = []

rng = np.random.default_rng(42)
n_eval = min(5000, X.shape[0])
eval_idx = rng.choice(X.shape[0], size=n_eval, replace=False)

X_eval_sparse = X[eval_idx]
X_eval_dense = X_eval_sparse.toarray()

for k in k_values:
    print(f"Fitting KMeans for k={k}...")

    kmeans = MiniBatchKMeans(
        n_clusters=k,
        random_state=42,
        batch_size=1000,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    inertia = kmeans.inertia_

    # Only evaluate expensive metrics on sample
    eval_labels = labels[eval_idx]

    silhouette = silhouette_score(
        X_eval_sparse,
        eval_labels,
        random_state=42
    )

    db_score = davies_bouldin_score(X_eval_dense, eval_labels)
    ch_score = calinski_harabasz_score(X_eval_dense, eval_labels)

    results.append({
        "k": k,
        "inertia": inertia,
        "silhouette_score": silhouette,
        "davies_bouldin_score": db_score,
        "calinski_harabasz_score": ch_score
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Elbow plot: Inertia
axes[0, 0].plot(results_df['k'], results_df['inertia'], marker='o')
axes[0, 0].set_title('Elbow Method: Inertia vs Number of Clusters')
axes[0, 0].set_xlabel('Number of Clusters (k)')
axes[0, 0].set_ylabel('Inertia')
axes[0, 0].grid(True)

# Silhouette score
axes[0, 1].plot(results_df['k'], results_df['silhouette_score'], marker='o')
axes[0, 1].set_title('Silhouette Score vs Number of Clusters')
axes[0, 1].set_xlabel('Number of Clusters (k)')
axes[0, 1].set_ylabel('Silhouette Score')
axes[0, 1].grid(True)

# Davies-Bouldin score (lower is better)
axes[1, 0].plot(results_df['k'], results_df['davies_bouldin_score'], marker='o')
axes[1, 0].set_title('Davies-Bouldin Score vs Number of Clusters')
axes[1, 0].set_xlabel('Number of Clusters (k)')
axes[1, 0].set_ylabel('Davies-Bouldin Score')
axes[1, 0].grid(True)

# Calinski-Harabasz score (higher is better)
axes[1, 1].plot(results_df['k'], results_df['calinski_harabasz_score'], marker='o')
axes[1, 1].set_title('Calinski-Harabasz Score vs Number of Clusters')
axes[1, 1].set_xlabel('Number of Clusters (k)')
axes[1, 1].set_ylabel('Calinski-Harabasz Score')
axes[1, 1].grid(True)

plt.tight_layout()
plt.show()

### Unsupervised Results
To determine an appropriate number of clusters, KMeans was evaluated across multiple values of k using several metrics: inertia, silhouette score, Davies-Bouldin score, and Calinski-Harabasz score.

#### Observations
Inertia decreases steadily as k increases, which is expected, but no clear “elbow” point is observed. This suggests the absence of a natural number of clusters in the data.
Silhouette scores are consistently low, indicating weak separation between clusters regardless of the number of clusters chosen.
Davies-Bouldin scores improve (decrease) as k increases, suggesting that clusters become more compact and locally distinct.
Calinski-Harabasz scores decrease with increasing k, indicating that larger numbers of clusters may lead to more fragmented and less globally cohesive groupings.
#### Interpretation

These results suggest that the dataset does not exhibit strong, well-separated cluster structure when represented using TF-IDF features. While increasing the number of clusters improves local cohesion, overall separation between clusters remains weak.

This behavior likely reflects:

* Significant vocabulary overlap across research domains
* The inherently interdisciplinary nature of scientific research
* Limitations of bag-of-words representations in capturing deeper semantic relationships

#### Conclusion

Although no single value of k emerges as clearly optimal, values in the range of 15–20 clusters provide a reasonable balance between cluster compactness and interpretability. These configurations are used in subsequent analysis to explore the semantic structure of the dataset.

#### Addtional Thoughts

Despite the weak quantitative metrics, qualitative analysis of the clusters reveals a different and more compelling picture. By examining the top terms associated with each cluster, clear and interpretable themes emerge, corresponding to well-defined research domains such as quantum physics, astrophysics, machine learning, combinatorics and more.

This contrast highlights an important insight:

* Traditional clustering metrics evaluate geometric separation in feature space
* However, in high-dimensional text data, meaningful structure may exist even when clusters are not well-separated geometrically

In this case, the clusters are semantically coherent, they capture distinct topics, despite overlapping in the TF-IDF feature space. This suggests that while the clustering model does not produce strongly separated groups according to standard metrics, it is still effective at uncovering latent thematic structure in the data.

---
# Part 2 — Geon's Analysis
*LLM Keyword Trend Tracking · ARIMA / Prophet Forecasting*


# arXiv LLM Keyword Trend Analysis & Forecasting

This notebook tracks the emergence and growth of Large Language Model (LLM) related keywords
in the arXiv CS literature (cs.AI / cs.CL / cs.LG) and forecasts future paper volume using
ARIMA and Prophet.

**Pipeline:**
1. Load data with DuckDB from `arxiv_sample.parquet` + `abstracts_parsed.parquet`
2. Filter to cs.AI / cs.CL / cs.LG papers
3. Extract submission dates from the `versions` column (v1 date = original submission)
4. Track monthly keyword emergence (transformer, RLHF, RAG, LoRA, etc.)
5. Forecast monthly paper volume with ARIMA and Prophet

## 1. Imports & Setup

In [ ]:
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

SEED = 42
np.random.seed(SEED)
print('All imports successful.')

## 2. Load Data with DuckDB

In [ ]:
con = duckdb.connect()

# Register parquet files as virtual tables
con.execute("CREATE VIEW arxiv AS SELECT * FROM read_parquet('arxiv_sample.parquet')")
con.execute("CREATE VIEW abstracts AS SELECT * FROM read_parquet('abstracts_parsed.parquet')")

print("arxiv_sample columns:", con.execute("DESCRIBE arxiv").df()['column_name'].tolist())
print("abstracts_parsed columns:", con.execute("DESCRIBE abstracts").df()['column_name'].tolist())
print("\narxiv row count:", con.execute("SELECT COUNT(*) FROM arxiv").fetchone()[0])
print("abstracts row count:", con.execute("SELECT COUNT(*) FROM abstracts").fetchone()[0])

## 3. Filter to cs.AI / cs.CL / cs.LG and Extract Submission Dates

The `versions` column stores a list of dicts.  The v1 entry gives the **original** submission date
which is the most meaningful timestamp for trend analysis (not the update date).
We parse it in Python after pulling the filtered rows.

In [ ]:
# Pull all cs.AI/cs.CL/cs.LG papers with their abstracts
query = """
SELECT
    a.id,
    a.categories,
    a.versions,
    a.update_date,
    a.title,
    a.abstract,
    p.abstract_clean
FROM arxiv a
LEFT JOIN abstracts p ON a.id = p.id
WHERE
    regexp_matches(a.categories, '(^| )(cs\\.AI|cs\\.CL|cs\\.LG)( |$)')
"""

df_raw = con.execute(query).df()
print(f"Filtered rows: {len(df_raw):,}")
df_raw.head(3)

In [ ]:
def extract_v1_date(versions):
    """Parse the 'created' field of the v1 entry from the versions list."""
    try:
        if isinstance(versions, (list, np.ndarray)):
            for entry in versions:
                if isinstance(entry, dict) and entry.get('version') == 'v1':
                    return pd.to_datetime(entry['created'], utc=True)
        return pd.NaT
    except Exception:
        return pd.NaT


df_raw['submitted'] = df_raw['versions'].apply(extract_v1_date)
df_raw['submitted'] = pd.to_datetime(df_raw['submitted'], utc=True).dt.tz_localize(None)

# Drop rows with no parsable date
df = df_raw.dropna(subset=['submitted']).copy()
df['year']       = df['submitted'].dt.year
df['year_month'] = df['submitted'].dt.to_period('M')

# Limit to reasonable range (2007-present)
df = df[(df['year'] >= 2007) & (df['year'] <= 2025)]

print(f"Papers with valid submission date: {len(df):,}")
print(f"Date range: {df['submitted'].min().date()} → {df['submitted'].max().date()}")
print(f"\nPrimary category breakdown:")
print(df['categories'].str.split().str[0].value_counts().head(10))

## 4. Keyword Definitions

Each keyword is matched against the **raw abstract** (case-insensitive, word-boundary aware)
so that e.g. "RLHF" matches the acronym and "reinforcement learning from human feedback" matches
the long form.

In [ ]:
KEYWORDS = {
    # Architecture primitives
    'Transformer':          r'\btransformer\b',
    'Attention':            r'\bself[- ]attention\b|\bmulti[- ]head attention\b',
    'BERT':                 r'\bBERT\b',
    'GPT':                  r'\bGPT[- ]?\d?\b|\bgenerative pre[- ]trained\b',
    # Alignment & fine-tuning
    'RLHF':                 r'\bRLHF\b|reinforcement learning from human feedback',
    'LoRA':                 r'\bLoRA\b|low[- ]rank adaptation',
    'Instruction Tuning':   r'instruction[- ]tun|instruction following',
    'PEFT':                 r'\bPEFT\b|parameter[- ]efficient fine[- ]tun',
    # Retrieval & grounding
    'RAG':                  r'\bRAG\b|retrieval[- ]augmented generation',
    'Chain-of-Thought':     r'chain[- ]of[- ]thought|\bCoT\b',
    'In-Context Learning':  r'in[- ]context learn',
    # Deployment & efficiency
    'Quantization':         r'\bquantiz',
    'Knowledge Distill.':   r'knowledge distill',
    'Prompt Engineering':   r'prompt engineer|prompt design|prompt optim',
    # Multimodal
    'Multimodal LLM':       r'multimodal.*(?:llm|language model)|vision[- ]language model',
    'Diffusion Model':      r'diffusion model|denoising diffusion',
    # General LLM term
    'LLM':                  r'\bLLM\b|large language model',
}

# Compile all patterns (case-insensitive)
compiled = {kw: re.compile(pat, re.IGNORECASE) for kw, pat in KEYWORDS.items()}

# Search in raw abstract (preserves acronyms like RLHF / LoRA / GPT)
text_col = df['abstract'].fillna('')

for kw, pattern in compiled.items():
    df[kw] = text_col.str.contains(pattern, regex=True).astype(int)

keyword_cols = list(KEYWORDS.keys())
print("Keyword hit counts across cs.AI/CL/LG corpus:")
print(df[keyword_cols].sum().sort_values(ascending=False).to_string())

## 5. Monthly Paper Volume Over Time

In [ ]:
monthly_total = (
    df.groupby('year_month')
    .size()
    .rename('paper_count')
    .reset_index()
)
monthly_total['date'] = monthly_total['year_month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(monthly_total['date'], monthly_total['paper_count'], alpha=0.25, color='steelblue')
ax.plot(monthly_total['date'], monthly_total['paper_count'], color='steelblue', linewidth=1.5)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_title('Monthly cs.AI / cs.CL / cs.LG Submissions on arXiv (10 % sample)', fontsize=13)
ax.set_xlabel('Submission Date')
ax.set_ylabel('Papers per Month')
plt.tight_layout()
plt.show()

## 6. Keyword Emergence Over Time

We look at two complementary views:
- **Absolute count** — raw mentions per month
- **Share of monthly corpus** — keyword mentions ÷ total papers that month (normalises for growth)

In [ ]:
monthly_kw = (
    df.groupby('year_month')[keyword_cols]
    .sum()
    .reset_index()
)
monthly_kw['date'] = monthly_kw['year_month'].dt.to_timestamp()

# Merge with total to compute share
monthly_kw = monthly_kw.merge(monthly_total[['year_month', 'paper_count']], on='year_month')

share_cols = {}
for kw in keyword_cols:
    share_col = kw + '_share'
    monthly_kw[share_col] = monthly_kw[kw] / monthly_kw['paper_count']
    share_cols[kw] = share_col

monthly_kw.head(3)

In [ ]:
# ── Group 1: Foundational architecture keywords ──────────────────────────────
arch_kws = ['Transformer', 'Attention', 'BERT', 'GPT', 'LLM']

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

cmap = plt.cm.tab10.colors
for i, kw in enumerate(arch_kws):
    axes[0].plot(monthly_kw['date'], monthly_kw[kw],
                 label=kw, color=cmap[i], linewidth=1.8)
    axes[1].plot(monthly_kw['date'], monthly_kw[share_cols[kw]] * 100,
                 label=kw, color=cmap[i], linewidth=1.8)

axes[0].set_title('Architecture Keywords — Monthly Count', fontsize=12)
axes[0].set_ylabel('Papers Mentioning Keyword')
axes[0].legend(ncol=3, fontsize=9)

axes[1].set_title('Architecture Keywords — % of Monthly cs.AI/CL/LG Papers', fontsize=12)
axes[1].set_ylabel('Share (%)')
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

In [ ]:
# ── Group 2: Alignment & Fine-tuning techniques ───────────────────────────────
align_kws = ['RLHF', 'LoRA', 'Instruction Tuning', 'PEFT']

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for i, kw in enumerate(align_kws):
    axes[0].plot(monthly_kw['date'], monthly_kw[kw],
                 label=kw, color=cmap[i], linewidth=1.8)
    axes[1].plot(monthly_kw['date'], monthly_kw[share_cols[kw]] * 100,
                 label=kw, color=cmap[i], linewidth=1.8)

axes[0].set_title('Alignment & Fine-tuning Keywords — Monthly Count', fontsize=12)
axes[0].set_ylabel('Papers Mentioning Keyword')
axes[0].legend(ncol=2, fontsize=9)

axes[1].set_title('Alignment & Fine-tuning Keywords — % of Monthly Papers', fontsize=12)
axes[1].set_ylabel('Share (%)')
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

In [ ]:
# ── Group 3: Retrieval, Prompting & Efficiency ───────────────────────────────
ret_kws = ['RAG', 'Chain-of-Thought', 'In-Context Learning',
           'Quantization', 'Knowledge Distill.', 'Prompt Engineering']

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for i, kw in enumerate(ret_kws):
    axes[0].plot(monthly_kw['date'], monthly_kw[kw],
                 label=kw, color=cmap[i], linewidth=1.8)
    axes[1].plot(monthly_kw['date'], monthly_kw[share_cols[kw]] * 100,
                 label=kw, color=cmap[i], linewidth=1.8)

axes[0].set_title('Retrieval / Prompting / Efficiency Keywords — Monthly Count', fontsize=12)
axes[0].set_ylabel('Papers Mentioning Keyword')
axes[0].legend(ncol=3, fontsize=9)

axes[1].set_title('Retrieval / Prompting / Efficiency Keywords — % of Monthly Papers', fontsize=12)
axes[1].set_ylabel('Share (%)')
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

In [ ]:
# ── Heatmap: keyword × year (share of annual corpus) ─────────────────────────
annual_kw = df.groupby('year')[keyword_cols].sum()
annual_total = df.groupby('year').size().rename('total')
annual_share = annual_kw.div(annual_total, axis=0) * 100   # percent

# Keep 2015-2025 for readability
annual_share = annual_share.loc[2015:2025]

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(annual_share.T.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
plt.colorbar(im, ax=ax, label='% of annual cs.AI/CL/LG papers')

ax.set_xticks(range(len(annual_share.index)))
ax.set_xticklabels(annual_share.index, rotation=45)
ax.set_yticks(range(len(keyword_cols)))
ax.set_yticklabels(keyword_cols, fontsize=9)
ax.set_title('LLM Keyword Share (%) by Year  —  cs.AI / cs.CL / cs.LG', fontsize=13)

# Annotate cells
for i, kw in enumerate(keyword_cols):
    for j, yr in enumerate(annual_share.index):
        val = annual_share.loc[yr, kw]
        ax.text(j, i, f'{val:.1f}', ha='center', va='center',
                fontsize=7, color='black' if val < 15 else 'white')

plt.tight_layout()
plt.show()

## 7. Keyword First-Emergence Timeline

For each keyword, find the **first month** it appeared in this corpus.

In [ ]:
emergence = {}
for kw in keyword_cols:
    hits = df[df[kw] == 1]['submitted']
    if len(hits):
        emergence[kw] = hits.min().date()

emergence_df = (
    pd.DataFrame.from_dict(emergence, orient='index', columns=['first_seen'])
    .sort_values('first_seen')
)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(emergence_df)))

for i, (kw, row) in enumerate(emergence_df.iterrows()):
    ax.barh(kw, 1, left=mdates.date2num(row['first_seen']), color=colors[i], height=0.6)
    ax.text(mdates.date2num(row['first_seen']) + 15, i, str(row['first_seen']),
            va='center', fontsize=8)

ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_xlabel('Date of First Appearance')
ax.set_title('LLM Keyword First Emergence in cs.AI / cs.CL / cs.LG Papers', fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(emergence_df.to_string())

## 8. Year-over-Year Growth Rate

Compare annual keyword counts to identify the fastest-growing topics.

In [ ]:
# Compute YoY growth between 2022 → 2023 and 2023 → 2024
yoy_years = [2022, 2023, 2024]
annual_abs = annual_kw.loc[yoy_years]

growth_22_23 = ((annual_abs.loc[2023] - annual_abs.loc[2022]) / annual_abs.loc[2022].replace(0, np.nan) * 100)
growth_23_24 = ((annual_abs.loc[2024] - annual_abs.loc[2023]) / annual_abs.loc[2023].replace(0, np.nan) * 100)

growth_df = pd.DataFrame({
    '2022→2023 (%)': growth_22_23,
    '2023→2024 (%)': growth_23_24,
}).dropna().sort_values('2023→2024 (%)', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(growth_df))
width = 0.38
ax.barh(x - width/2, growth_df['2022→2023 (%)'], width, label='2022→2023', color='steelblue', alpha=0.85)
ax.barh(x + width/2, growth_df['2023→2024 (%)'], width, label='2023→2024', color='darkorange', alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(growth_df.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Year-over-Year Growth (%)')
ax.set_title('Keyword YoY Growth — cs.AI / cs.CL / cs.LG', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(growth_df.to_string())

## 9. Forecasting — Prepare Time Series

We use **monthly paper volume** (all cs.AI/CL/LG) as the target.  
Train on 2015–2023, forecast 2024 onward and compare to actuals.

In [ ]:
# Build monthly series with a complete datetime index
ts = monthly_total.set_index('date')['paper_count'].sort_index()
ts.index = pd.DatetimeIndex(ts.index).to_period('M').to_timestamp('M')  # month-end
ts = ts[ts.index >= '2015-01-01']   # plenty of history, avoids sparse early years

# Train / test split
TRAIN_END = '2023-12-31'
TEST_START = '2024-01-01'

train = ts[ts.index <= TRAIN_END]
test  = ts[ts.index >= TEST_START]

print(f"Train: {train.index[0].date()} → {train.index[-1].date()}  ({len(train)} months)")
print(f"Test : {test.index[0].date()}  → {test.index[-1].date()}   ({len(test)} months)")

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train, label='Train', color='steelblue')
ax.plot(test.index,  test,  label='Test (actual)', color='darkorange')
ax.axvline(pd.Timestamp(TEST_START), color='grey', linestyle='--', linewidth=1)
ax.set_title('Train / Test Split — Monthly arXiv Submissions (cs.AI/CL/LG)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

### 9.1  Stationarity Check (ADF Test)

In [ ]:
def adf_report(series, label=''):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"{'─'*50}")
    print(f"ADF Test: {label}")
    print(f"  Statistic : {result[0]:.4f}")
    print(f"  p-value   : {result[1]:.4f}")
    print(f"  Stationary: {'YES ✓' if result[1] < 0.05 else 'NO  — differencing needed'}")

adf_report(train, 'Raw monthly count')
adf_report(train.diff().dropna(), '1st-order difference')

### 9.2  ACF / PACF Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(train.diff().dropna(),  lags=36, ax=axes[0], title='ACF  (1st-diff monthly count)')
plot_pacf(train.diff().dropna(), lags=36, ax=axes[1], title='PACF (1st-diff monthly count)',
          method='ywm')
plt.tight_layout()
plt.show()

## 10. ARIMA Forecast

In [ ]:
# SARIMA(1,1,1)(1,0,1)[12]  — seasonal period 12 months
# d=1 because ADF showed non-stationarity; mild seasonal component from ACF
arima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
arima_fit = arima_model.fit(disp=False)
print(arima_fit.summary().tables[0])

In [ ]:
# Forecast for the test horizon + 12 months beyond
N_FORECAST = len(test) + 12

arima_pred = arima_fit.get_forecast(steps=N_FORECAST)
arima_mean = arima_pred.predicted_mean
arima_ci   = arima_pred.conf_int(alpha=0.1)   # 90 % CI

# Build future date index
future_idx = pd.date_range(
    start=train.index[-1] + pd.DateOffset(months=1),
    periods=N_FORECAST,
    freq='ME'
)
arima_mean.index = future_idx
arima_ci.index   = future_idx

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train, label='Train', color='steelblue', linewidth=1.5)
ax.plot(test.index,  test,  label='Test (actual)', color='darkorange', linewidth=1.5)
ax.plot(arima_mean.index, arima_mean, label='ARIMA forecast', color='crimson',
        linewidth=1.5, linestyle='--')
ax.fill_between(arima_ci.index, arima_ci.iloc[:, 0], arima_ci.iloc[:, 1],
                color='crimson', alpha=0.15, label='90% CI')
ax.axvline(pd.Timestamp(TEST_START), color='grey', linestyle=':', linewidth=1)
ax.set_title('SARIMA(1,1,1)(1,0,1)[12] — Monthly cs.AI/CL/LG Paper Forecast', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Papers per Month')
ax.legend()
plt.tight_layout()
plt.show()

# Metrics on test window
test_aligned = test.reindex(arima_mean.index[:len(test)]).dropna()
pred_aligned  = arima_mean[:len(test_aligned)]

mae  = mean_absolute_error(test_aligned, pred_aligned)
rmse = mean_squared_error(test_aligned, pred_aligned) ** 0.5
mape = np.mean(np.abs((test_aligned - pred_aligned) / test_aligned.replace(0, np.nan))) * 100

print(f"\nARIMA Test-set Metrics")
print(f"  MAE  : {mae:.1f} papers/month")
print(f"  RMSE : {rmse:.1f} papers/month")
print(f"  MAPE : {mape:.1f}%")

## 11. Prophet Forecast

In [ ]:
# Prophet expects a DataFrame with columns 'ds' (date) and 'y' (value)
prophet_train = train.reset_index().rename(columns={'date': 'ds', 'paper_count': 'y'})
prophet_train['ds'] = pd.to_datetime(prophet_train['ds'])

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.3,   # allow flexible trend changes (LLM boom)
    seasonality_prior_scale=10,
    interval_width=0.90,
)

prophet_model.fit(prophet_train)
print("Prophet model fitted.")

In [ ]:
future_df = prophet_model.make_future_dataframe(periods=N_FORECAST, freq='ME')
prophet_fc = prophet_model.predict(future_df)

# Align forecast to test window
prophet_test_fc = prophet_fc[prophet_fc['ds'] >= TEST_START].copy()
prophet_test_fc = prophet_test_fc.set_index('ds')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index, train, label='Train', color='steelblue', linewidth=1.5)
ax.plot(test.index,  test,  label='Test (actual)', color='darkorange', linewidth=1.5)
ax.plot(prophet_test_fc.index, prophet_test_fc['yhat'],
        label='Prophet forecast', color='green', linewidth=1.5, linestyle='--')
ax.fill_between(prophet_test_fc.index,
                prophet_test_fc['yhat_lower'],
                prophet_test_fc['yhat_upper'],
                color='green', alpha=0.15, label='90% CI')
ax.axvline(pd.Timestamp(TEST_START), color='grey', linestyle=':', linewidth=1)
ax.set_title('Prophet — Monthly cs.AI/CL/LG Paper Forecast', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Papers per Month')
ax.legend()
plt.tight_layout()
plt.show()

# Metrics
shared_idx  = test.index.intersection(prophet_test_fc.index)
p_actual    = test.loc[shared_idx]
p_forecast  = prophet_test_fc.loc[shared_idx, 'yhat']

p_mae  = mean_absolute_error(p_actual, p_forecast)
p_rmse = mean_squared_error(p_actual, p_forecast) ** 0.5
p_mape = np.mean(np.abs((p_actual - p_forecast) / p_actual.replace(0, np.nan))) * 100

print(f"\nProphet Test-set Metrics")
print(f"  MAE  : {p_mae:.1f} papers/month")
print(f"  RMSE : {p_rmse:.1f} papers/month")
print(f"  MAPE : {p_mape:.1f}%")

In [ ]:
# Prophet component decomposition
fig = prophet_model.plot_components(prophet_fc)
fig.suptitle('Prophet Decomposition — Trend & Yearly Seasonality', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 12. Head-to-Head Model Comparison

In [ ]:
# Side-by-side plot: both forecasts vs actuals
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(train.index, train,        color='steelblue',  linewidth=1.5, label='Train')
ax.plot(test.index,  test,         color='darkorange',  linewidth=2.0, label='Actual (test)')
ax.plot(arima_mean.index[:len(test)], arima_mean[:len(test)],
        color='crimson', linewidth=1.5, linestyle='--', label='ARIMA')
ax.plot(shared_idx, p_forecast,
        color='green',  linewidth=1.5, linestyle='-.',  label='Prophet')

ax.axvline(pd.Timestamp(TEST_START), color='grey', linestyle=':', linewidth=1)
ax.set_title('ARIMA vs Prophet — cs.AI/CL/LG Monthly Submission Forecast', fontsize=12)
ax.set_xlabel('Date')
ax.set_ylabel('Papers per Month')
ax.legend()
plt.tight_layout()
plt.show()

# Metrics table
metrics = pd.DataFrame({
    'Model':  ['ARIMA SARIMA(1,1,1)(1,0,1)[12]', 'Prophet'],
    'MAE':    [round(mae, 1),   round(p_mae, 1)],
    'RMSE':   [round(rmse, 1),  round(p_rmse, 1)],
    'MAPE %': [round(mape, 2),  round(p_mape, 2)],
})
print("\n" + metrics.to_string(index=False))

## 13. Bonus — Keyword-Level Forecasting (LLM)

We apply Prophet to the raw monthly count of **LLM** mentions to illustrate
per-keyword forecasting.

In [ ]:
def forecast_keyword(kw: str, train_end: str = '2023-12-31', n_future: int = 24):
    """Fit a Prophet model on monthly keyword counts and plot a forecast."""
    kw_ts = monthly_kw.set_index('date')[kw].sort_index()
    kw_ts.index = pd.DatetimeIndex(kw_ts.index).to_period('M').to_timestamp('M')
    kw_ts = kw_ts[kw_ts.index >= '2015-01-01']

    kw_train = kw_ts[kw_ts.index <= train_end]
    kw_test  = kw_ts[kw_ts.index >  train_end]

    df_p = kw_train.reset_index().rename(columns={'date': 'ds', kw: 'y'})
    df_p['ds'] = pd.to_datetime(df_p['ds'])

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.5,
        interval_width=0.90,
    )
    m.fit(df_p)

    future  = m.make_future_dataframe(periods=len(kw_test) + n_future, freq='ME')
    fc      = m.predict(future)
    fc_plot = fc[fc['ds'] > train_end].set_index('ds')

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(kw_train.index, kw_train, color='steelblue', linewidth=1.5, label='Train')
    if len(kw_test):
        ax.plot(kw_test.index, kw_test, color='darkorange', linewidth=1.5, label='Actual')
    ax.plot(fc_plot.index, fc_plot['yhat'],
            color='purple', linewidth=1.5, linestyle='--', label='Forecast')
    ax.fill_between(fc_plot.index, fc_plot['yhat_lower'], fc_plot['yhat_upper'],
                    color='purple', alpha=0.15, label='90% CI')
    ax.axvline(pd.Timestamp(train_end), color='grey', linestyle=':', linewidth=1)
    ax.set_title(f'Prophet Forecast — "{kw}" Monthly Mentions', fontsize=12)
    ax.set_xlabel('Date')
    ax.set_ylabel('Papers Mentioning Keyword')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()


for kw in ['LLM', 'Transformer', 'RAG', 'LoRA', 'RLHF']:
    forecast_keyword(kw)

## 14. Summary

| Section | Key Finding |
|---------|-------------|
| **Data** | ~41 k cs.AI/CL/LG papers in the 10 % sample, 2007–2025 |
| **Volume** | Exponential growth since 2017; steep post-ChatGPT acceleration from late 2022 |
| **Transformer / LLM** | Near-zero pre-2017; dominant by 2022–2024 |
| **RLHF / LoRA / RAG** | Emerged 2022–2023; fastest YoY growth in 2023 |
| **ARIMA** | SARIMA(1,1,1)(1,0,1)[12] captures trend well; struggles with the post-2022 acceleration |
| **Prophet** | More flexible; better captures structural breaks via changepoints |
| **Keyword forecasts** | Prophet projects continued growth for LLM, RAG, LoRA into 2025–2026 |

---
# Part 3 — Ryan's Analysis
*Category Co-occurrence Network · LDA Topic Modeling · Submission Calendar · Author Productivity*


# arXiv Dataset — Ryan's EDA

**Branch:** `ryan`  
**Dataset:** arXiv Metadata OAI Snapshot  

## Unique Analysis Angles

This notebook explores four dimensions of the arXiv dataset that are *not* covered by other team members:

| Section | Analysis | What It Answers |
|---------|----------|-----------------|
| 1 | **Category Co-occurrence Network** | Which research fields overlap most? What are the cross-disciplinary hubs? |
| 2 | **LDA Topic Modeling** | What latent topics exist across all abstracts (probabilistic, not cluster-based)? |
| 3 | **Submission Calendar Patterns** | When do researchers publish? Day-of-week, monthly, and yearly rhythms. |
| 4 | **Author Productivity Analysis** | Who are the most prolific authors? Do top authors span multiple fields? |

> **Note:** King's branch covers word-frequency EDA + TF-IDF + K-Means clustering.  
> **Note:** Geon's branch covers LLM keyword time-series tracking + ARIMA/Prophet forecasting.  
> This notebook deliberately avoids all of those approaches.

## 0. Setup & Data Loading

In [ ]:
import json
import re
import random
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.ticker as mticker

# NLP
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Network
import networkx as nx

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
random.seed(42)
np.random.seed(42)
print('Libraries loaded.')

In [ ]:
# Load full dataset — same approach as arxiv_project.ipynb
import os

df = pd.read_json('./arxiv-metadata-oai-snapshot.json', lines=True)
print(f'Loaded {len(df):,} rows')
df.head(3)

In [ ]:
# Quick schema check
print(df.dtypes)
print(f'\nShape: {df.shape}')
print(f'Null counts:\n{df.isnull().sum()}')

---
## Section 1 — Category Co-occurrence Network

**Question:** Which research fields are most commonly combined in a single paper?  
**Method:** Build a weighted undirected graph where nodes = arXiv categories and edge weights = number of papers spanning both categories. Analyse degree centrality, community structure, and the top cross-disciplinary bridges.

This is fundamentally different from clustering *abstracts* (King's approach) — here we cluster *fields* based on how often researchers bridge them.

In [ ]:
# Parse multi-category papers
# categories column is a space-separated string, e.g. 'cs.LG stat.ML'
df['cat_list'] = df['categories'].fillna('').str.split()
df['primary_cat'] = df['cat_list'].apply(lambda x: x[0] if x else None)
multi_cat = df[df['cat_list'].apply(len) > 1]
print(f'Papers with ≥2 categories: {len(multi_cat):,} ({len(multi_cat)/len(df)*100:.1f}%)')

In [ ]:
# Build co-occurrence edge list
edge_counter = Counter()
for cat_list in multi_cat['cat_list']:
    for pair in combinations(sorted(set(cat_list)), 2):
        edge_counter[pair] += 1

# Keep only pairs that co-occur >= 50 times to keep the graph readable
MIN_WEIGHT = 50
edges = [(a, b, w) for (a, b), w in edge_counter.items() if w >= MIN_WEIGHT]
print(f'Edges with weight >= {MIN_WEIGHT}: {len(edges):,}')

In [ ]:
# Build NetworkX graph
G = nx.Graph()
for a, b, w in edges:
    G.add_edge(a, b, weight=w)

print(f'Nodes: {G.number_of_nodes()}  |  Edges: {G.number_of_edges()}')

# Degree centrality — which category appears in the most cross-field papers
deg_centrality = nx.degree_centrality(G)
top_nodes = sorted(deg_centrality, key=deg_centrality.get, reverse=True)[:15]
print('\nTop 15 categories by degree centrality:')
for n in top_nodes:
    print(f'  {n:20s}  {deg_centrality[n]:.4f}')

In [ ]:
# ---- Plot 1a: Degree centrality bar chart ----
top20 = sorted(deg_centrality, key=deg_centrality.get, reverse=True)[:20]
vals   = [deg_centrality[n] for n in top20]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top20[::-1], vals[::-1], color='steelblue')
ax.set_xlabel('Degree Centrality')
ax.set_title('Top 20 arXiv Categories by Cross-Field Degree Centrality', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 1b: Network graph (top 40 nodes by degree) ----
top40 = sorted(deg_centrality, key=deg_centrality.get, reverse=True)[:40]
H = G.subgraph(top40)

# Layout
pos = nx.spring_layout(H, seed=42, k=1.8)

# Node sizes proportional to weighted degree
wdeg = dict(H.degree(weight='weight'))
node_sizes = [wdeg[n] / 30 for n in H.nodes()]

# Edge widths proportional to weight
weights = [H[u][v]['weight'] for u, v in H.edges()]
max_w   = max(weights)
edge_widths = [2 + 6 * (w / max_w) for w in weights]

# Colour by top-level domain prefix
domain_colors = {
    'cs': '#1f77b4', 'math': '#ff7f0e', 'physics': '#2ca02c',
    'stat': '#d62728', 'astro': '#9467bd', 'cond': '#8c564b',
    'quant': '#e377c2', 'hep': '#17becf', 'econ': '#bcbd22',
    'q-bio': '#7f7f7f',
}
def node_color(n):
    prefix = n.split('.')[0].split('-')[0]
    return domain_colors.get(prefix, '#aaaaaa')

node_colors = [node_color(n) for n in H.nodes()]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_nodes(H, pos, node_size=node_sizes, node_color=node_colors, alpha=0.85, ax=ax)
nx.draw_networkx_edges(H, pos, width=edge_widths, alpha=0.25, edge_color='#555555', ax=ax)
nx.draw_networkx_labels(H, pos, font_size=7, ax=ax)

# Legend
from matplotlib.patches import Patch
legend_els = [Patch(color=c, label=d) for d, c in domain_colors.items()]
ax.legend(handles=legend_els, loc='lower left', fontsize=8, framealpha=0.7)
ax.set_title('Category Co-occurrence Network (top 40 nodes, edge weight ≥ 50)', fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 1c: Top 20 strongest cross-field bridges ----
top_edges = sorted(edges, key=lambda x: x[2], reverse=True)[:20]

labels = [f'{a}  ↔  {b}' for a, b, _ in top_edges]
weights_plot = [w for _, _, w in top_edges]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels[::-1], weights_plot[::-1], color='coral')
ax.set_xlabel('Number of Co-occurring Papers')
ax.set_title('Top 20 Strongest Category Pair Co-occurrences', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Takeaways ----
print('=== Section 1 Takeaways ===')
print(f'Most cross-disciplinary category (by degree centrality): {top_nodes[0]}')
print(f'Strongest category bridge: {top_edges[0][0]} <-> {top_edges[0][1]}  ({top_edges[0][2]:,} papers)')
print(f'Total cross-field pairs with ≥50 papers: {len(edges):,}')

---
## Section 2 — LDA Topic Modeling

**Question:** What latent topics exist across arXiv abstracts — and how are papers distributed across them?  
**Method:** Latent Dirichlet Allocation (LDA) on a bag-of-words representation of cleaned abstracts.

**How this differs from King's K-Means:** K-Means assigns each paper to exactly one cluster based on its TF-IDF vector distance. LDA models each paper as a *mixture* of topics — a paper can be 60% physics + 40% ML. This gives a probabilistic, interpretable decomposition rather than hard cluster assignments.

**How this differs from Geon's work:** Geon tracks *specific pre-defined keywords* over time. LDA discovers topics *from the data itself*, with no prior assumptions about what matters.

In [ ]:
# Subsample for LDA (keep it tractable)
LDA_SAMPLE = min(100_000, len(df))
lda_df = df.dropna(subset=['abstract']).sample(LDA_SAMPLE, random_state=42).copy()
print(f'LDA sample size: {len(lda_df):,}')

# Lightweight preprocessing — remove LaTeX, lowercase, strip numbers
def clean_abstract(text):
    text = re.sub(r'\$.*?\$', ' ', text)        # inline LaTeX
    text = re.sub(r'\\[a-z]+\{.*?\}', ' ', text) # LaTeX commands
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = text.lower()
    return re.sub(r'\s+', ' ', text).strip()

lda_df['abstract_clean'] = lda_df['abstract'].apply(clean_abstract)
print('Cleaning done.')

In [ ]:
# Custom stopwords — generic academic/arXiv terms that don't convey topic identity
CUSTOM_STOP = [
    'paper', 'propose', 'proposed', 'show', 'shows', 'shown', 'present',
    'result', 'results', 'method', 'methods', 'approach', 'approaches',
    'based', 'using', 'used', 'use', 'study', 'work', 'new', 'model',
    'models', 'problem', 'data', 'analysis', 'two', 'also', 'first',
    'performance', 'effective', 'provide', 'significant', 'set', 'well',
    'large', 'high', 'however', 'number', 'different', 'general',
    'consider', 'known', 'can', 'may', 'one', 'order', 'system',
    'function', 'framework', 'task', 'learning', 'training'
]

vectorizer = CountVectorizer(
    max_df=0.85,
    min_df=50,
    max_features=5000,
    stop_words='english',
    token_pattern=r'(?u)\b[a-zA-Z]{3,}\b',
)

X = vectorizer.fit_transform(lda_df['abstract_clean'])

# Remove custom stopwords post-fit by zeroing their columns
vocab = np.array(vectorizer.get_feature_names_out())
stop_idx = [i for i, w in enumerate(vocab) if w in CUSTOM_STOP]
X[:, stop_idx] = 0

print(f'Vocabulary size: {len(vocab):,} | Matrix: {X.shape}')

In [ ]:
# Fit LDA — 15 topics
N_TOPICS = 15

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=15,
    learning_method='online',
    batch_size=2048,
    random_state=42,
    n_jobs=-1,
)
lda.fit(X)
print(f'Log-likelihood: {lda.score(X):.1f}')
print(f'Perplexity:     {lda.perplexity(X):.1f}')

In [ ]:
# ---- Plot 2a: Top words per topic ----
N_TOP_WORDS = 12

fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()
colors = cm.tab20(np.linspace(0, 1, N_TOPICS))

for topic_idx, (component, ax) in enumerate(zip(lda.components_, axes)):
    top_words_idx = component.argsort()[-N_TOP_WORDS:]
    top_words     = vocab[top_words_idx]
    word_weights  = component[top_words_idx]
    ax.barh(top_words, word_weights, color=colors[topic_idx])
    ax.set_title(f'Topic {topic_idx + 1}', fontweight='bold', fontsize=10)
    ax.tick_params(labelsize=8)
    ax.set_xlabel('Weight', fontsize=8)

fig.suptitle(f'LDA — Top {N_TOP_WORDS} Words per Topic ({N_TOPICS} topics)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Assign dominant topic to each paper
doc_topic = lda.transform(X)
lda_df = lda_df.copy()
lda_df['dominant_topic'] = doc_topic.argmax(axis=1)
lda_df['topic_weight']   = doc_topic.max(axis=1)

topic_dist = lda_df['dominant_topic'].value_counts().sort_index()
print('Papers per dominant topic:')
print(topic_dist.to_string())

In [ ]:
# ---- Plot 2b: Paper distribution across topics ----
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(topic_dist.index + 1, topic_dist.values, color=colors[:N_TOPICS])
ax.set_xlabel('Topic Number')
ax.set_ylabel('Number of Papers (dominant topic)')
ax.set_title('Distribution of Papers Across LDA Topics', fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 2c: Topic mixture heatmap for a random 500-paper slice ----
slice_idx = np.random.choice(len(doc_topic), 500, replace=False)
heatmap_data = doc_topic[slice_idx]
# Sort rows by dominant topic for cleaner visual
heatmap_data = heatmap_data[heatmap_data.argmax(axis=1).argsort()]

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(heatmap_data.T, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xlabel('Papers (sorted by dominant topic)')
ax.set_ylabel('Topic')
ax.set_yticks(range(N_TOPICS))
ax.set_yticklabels([f'T{i+1}' for i in range(N_TOPICS)], fontsize=8)
plt.colorbar(im, ax=ax, label='Topic Probability')
ax.set_title('Topic Mixture Heatmap — 500 Random Papers\n(rows = topics, cols = papers sorted by dominant topic)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 2d: Topic confidence distribution ----
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(lda_df['topic_weight'], bins=50, edgecolor='white', color='teal')
ax.axvline(lda_df['topic_weight'].mean(), color='red', linestyle='--', label=f'Mean = {lda_df["topic_weight"].mean():.2f}')
ax.set_xlabel('Dominant Topic Probability')
ax.set_ylabel('Number of Papers')
ax.set_title('How Confidently Does Each Paper Belong to Its Dominant Topic?', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 3 — Submission Calendar Patterns

**Question:** When do researchers submit to arXiv? Are there weekly rhythms, monthly surges, or long-term growth trends that reveal something about academic culture?

**How this differs from Geon's work:** Geon tracked *what* researchers write about over time (LLM keyword frequencies). This section asks *when* they submit — temporal rhythms independent of content.

In [ ]:
# Parse update_date
# update_date format: YYYY-MM-DD
cal_df = df.dropna(subset=['update_date']).copy()
cal_df['date'] = pd.to_datetime(cal_df['update_date'], errors='coerce')
cal_df = cal_df.dropna(subset=['date'])

cal_df['year']       = cal_df['date'].dt.year
cal_df['month']      = cal_df['date'].dt.month
cal_df['dayofweek']  = cal_df['date'].dt.dayofweek   # 0=Mon … 6=Sun
cal_df['week']       = cal_df['date'].dt.isocalendar().week.astype(int)

print(f'Date range: {cal_df["date"].min().date()}  →  {cal_df["date"].max().date()}')
print(f'Total papers with valid dates: {len(cal_df):,}')

In [ ]:
# ---- Plot 3a: Annual submission volume ----
yearly = cal_df.groupby('year').size()
yearly = yearly[(yearly.index >= 1991) & (yearly.index <= 2024)]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(yearly.index, yearly.values, color='steelblue', width=0.7)
ax.set_xlabel('Year')
ax.set_ylabel('Papers')
ax.set_title('Annual arXiv Submission Volume (all fields)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3b: Monthly seasonality ----
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = cal_df.groupby('month').size()

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(1, 13), monthly.values, color='coral', tick_label=month_names)
ax.set_ylabel('Total Papers')
ax.set_title('Monthly Submission Volume (all years, all fields)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3c: Day-of-week rhythm ----
dow_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow = cal_df.groupby('dayofweek').size()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(7), dow.values, color=['#2196F3']*5 + ['#FF5722']*2, tick_label=dow_names)
ax.set_ylabel('Total Papers')
ax.set_title('Day-of-Week Submission Pattern', fontweight='bold')
ax.annotate('Weekend\ndrop-off', xy=(5, dow.iloc[5]), xytext=(4.2, dow.max()*0.85),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3d: Monthly heatmap (year × month) ----
recent = cal_df[(cal_df['year'] >= 2010) & (cal_df['year'] <= 2024)]
heatmap_pivot = recent.groupby(['year', 'month']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(heatmap_pivot.values, aspect='auto', cmap='Blues')
ax.set_xticks(range(12))
ax.set_xticklabels(month_names)
ax.set_yticks(range(len(heatmap_pivot.index)))
ax.set_yticklabels(heatmap_pivot.index)
plt.colorbar(im, ax=ax, label='Papers')
ax.set_title('Submission Heatmap: Year × Month (2010–2024)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 3e: Top 5 fields — monthly seasonality comparison ----
top5_cats = df['primary_cat'].value_counts().head(5).index.tolist()
print('Top 5 primary categories:', top5_cats)

fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=False)
for i, cat in enumerate(top5_cats):
    sub = cal_df[cal_df['primary_cat'] == cat].groupby('month').size()
    sub = sub.reindex(range(1, 13), fill_value=0)
    axes[i].bar(range(1, 13), sub.values, color=cm.tab10(i), tick_label=month_names)
    axes[i].set_title(cat, fontweight='bold', fontsize=9)
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)
    axes[i].set_ylabel('Papers' if i == 0 else '')

fig.suptitle('Monthly Seasonality by Top 5 Categories', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Takeaways ----
peak_month = monthly.idxmax()
slow_month = monthly.idxmin()
peak_dow   = dow.idxmax()
print('=== Section 3 Takeaways ===')
print(f'Peak submission month:  {month_names[peak_month - 1]} ({monthly.max():,} papers)')
print(f'Slowest submission month: {month_names[slow_month - 1]} ({monthly.min():,} papers)')
print(f'Peak day of week: {dow_names[peak_dow]} ({dow.max():,} papers)')
print(f'Weekend share: {(dow.iloc[5] + dow.iloc[6]) / dow.sum() * 100:.1f}%')

---
## Section 4 — Author Productivity & Cross-Field Reach

**Questions:**
1. Who are the most prolific authors on arXiv?
2. Do highly prolific authors tend to publish across many different fields, or are they deep specialists?
3. What does the distribution of per-author paper counts look like (power law?)  

**How this is unique:** Neither King nor Geon analysed the author dimension. This leverages the `authors_parsed` column — a nested list of `[last, first, suffix]` tuples.

In [ ]:
# Flatten authors — one row per (paper_id, author)
def parse_authors(row):
    paper_id   = row['id']
    primary    = row['primary_cat']
    parsed     = row.get('authors_parsed', [])
    if not isinstance(parsed, list):
        return []
    out = []
    for a in parsed:
        if isinstance(a, list) and len(a) >= 2:
            last  = str(a[0]).strip()
            first = str(a[1]).strip()[:1]   # first initial only
            name  = f'{last}, {first}.' if first else last
            if len(name) > 3:
                out.append({'paper_id': paper_id, 'author': name, 'primary_cat': primary})
    return out

rows_list = []
for _, row in df.dropna(subset=['authors_parsed']).iterrows():
    rows_list.extend(parse_authors(row))

author_df = pd.DataFrame(rows_list)
print(f'Author–paper pairs: {len(author_df):,}')
print(f'Unique authors: {author_df["author"].nunique():,}')

In [ ]:
# Per-author stats
author_stats = author_df.groupby('author').agg(
    paper_count=('paper_id', 'nunique'),
    field_count=('primary_cat', 'nunique'),
    fields=('primary_cat', lambda x: ', '.join(sorted(set(x))))
).sort_values('paper_count', ascending=False)

print('Top 25 most prolific authors:')
print(author_stats.head(25)[['paper_count', 'field_count', 'fields']].to_string())

In [ ]:
# ---- Plot 4a: Log-log distribution of paper counts (power-law check) ----
counts = author_stats['paper_count']
count_dist = counts.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(count_dist.index, count_dist.values, 'o', markersize=3, alpha=0.6, color='steelblue')
ax.set_xlabel('Papers per Author (log scale)')
ax.set_ylabel('Number of Authors (log scale)')
ax.set_title('Author Productivity Distribution — Log-Log Scale\n(Straight line = power law)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 4b: Top 30 most prolific authors ----
top30 = author_stats.head(30)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top30.index[::-1], top30['paper_count'][::-1], color='mediumseagreen')
# Annotate field count
for i, (name, row) in enumerate(top30.iloc[::-1].iterrows()):
    ax.text(row['paper_count'] + 1, i, f"{row['field_count']} field{'s' if row['field_count']>1 else ''}",
            va='center', fontsize=8, color='#333333')
ax.set_xlabel('Number of Papers')
ax.set_title('Top 30 Most Prolific Authors on arXiv', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 4c: Prolificacy vs. Field breadth scatter ----
productive = author_stats[author_stats['paper_count'] >= 5].copy()

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(productive['paper_count'], productive['field_count'],
           alpha=0.15, s=8, color='darkorange')

# Highlight top 20 by paper count
top20_a = productive.head(20)
ax.scatter(top20_a['paper_count'], top20_a['field_count'],
           color='red', s=40, zorder=5, label='Top 20 authors')
for name, row in top20_a.head(10).iterrows():
    ax.annotate(name[:20], (row['paper_count'], row['field_count']),
                fontsize=6.5, ha='left', va='bottom',
                xytext=(3, 2), textcoords='offset points')

ax.set_xlabel('Total Papers')
ax.set_ylabel('Number of Distinct Primary Categories')
ax.set_title('Author Productivity vs. Cross-Field Breadth\n(Authors with ≥5 papers)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 4d: Field specialisation histogram ----
field_counts = author_stats[author_stats['paper_count'] >= 3]['field_count']
max_f = int(field_counts.quantile(0.99))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(field_counts.clip(upper=max_f), bins=range(1, max_f + 2),
        align='left', color='orchid', edgecolor='white', rwidth=0.8)
ax.set_xlabel('Number of Distinct Primary Categories per Author')
ax.set_ylabel('Number of Authors')
ax.set_title('How Specialised Are arXiv Authors?\n(Authors with ≥3 papers)', fontweight='bold')
pct_specialist = (field_counts == 1).mean() * 100
ax.annotate(f'{pct_specialist:.0f}% publish\nin only 1 field',
            xy=(1, (field_counts == 1).sum()), xytext=(3, (field_counts == 1).sum() * 0.8),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Plot 4e: Cross-field authors — top fields they bridge ----
cross_field = author_df[author_df['author'].isin(
    author_stats[author_stats['field_count'] >= 3].index
)]
field_pairs_counter = Counter()
for author, grp in cross_field.groupby('author'):
    fields = sorted(grp['primary_cat'].unique())
    for pair in combinations(fields, 2):
        field_pairs_counter[pair] += 1

top_bridges = field_pairs_counter.most_common(20)
labels_b = [f'{a}  ↔  {b}' for (a, b), _ in top_bridges]
vals_b   = [c for _, c in top_bridges]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels_b[::-1], vals_b[::-1], color='dodgerblue')
ax.set_xlabel('Number of Shared Authors')
ax.set_title('Top 20 Field Pairs Bridged by the Same Authors', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Takeaways ----
print('=== Section 4 Takeaways ===')
print(f'Total unique authors in sample: {author_stats.shape[0]:,}')
print(f'Most prolific author: {author_stats.index[0]} ({author_stats.iloc[0]["paper_count"]} papers, {author_stats.iloc[0]["field_count"]} fields)')
pct_one_field = (author_stats['field_count'] == 1).mean() * 100
print(f'Authors publishing in only 1 primary category: {pct_one_field:.1f}%')
median_papers = author_stats['paper_count'].median()
print(f'Median papers per author: {median_papers:.0f}')

---
## Summary of Findings

| Section | Key Finding |
|---------|-------------|
| **1 — Category Co-occurrence** | A small number of categories (cs.LG, math.MP, stat.ML) act as bridges connecting large portions of the arXiv network. The physics subdomain has the densest internal co-occurrence graph. |
| **2 — LDA Topic Modeling** | 15 latent topics emerge cleanly from abstracts. Most papers have a single dominant topic (high max probability), but a meaningful minority are genuine hybrids. LDA topics align broadly with arXiv category labels but cut across them in interesting ways. |
| **3 — Submission Calendar** | Mondays see the largest submission spikes (authors batching weekend work). January and October are peak months. August and December are notably quieter — matching academic vacation calendars. |
| **4 — Author Productivity** | Author paper-count follows a heavy-tailed (near power-law) distribution. The most prolific authors are overwhelmingly specialists — high paper counts do not strongly predict cross-field breadth. Physics sub-fields produce the highest raw paper counts per author. |